# Preprocessing Step: Check and Remove Duplicate Rows
**Dataset:** BankChurners.csv
**Presenter part:** Duplicate Row Detection & Removal

This notebook walks through **why**, **how**, and **what actually happens**
when checking for duplicate rows — with a live demonstration, since the real
dataset happens to have zero duplicates (I inject some fake ones temporarily
just to prove the code genuinely detects and removes them).


## Step 1: Load the Dataset

In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

df = pd.read_csv("BankChurners.csv")
print("Original shape:", df.shape)
df.head()


Original shape: (10127, 23)


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


**What this does:** Loads the CSV into a DataFrame `df` — a table-like
structure where every row is one customer and every column is one feature
(e.g. `Customer_Age`, `Credit_Limit`, `Attrition_Flag`).

## Step 2: Check for Duplicate Rows in the Real Dataset

In [2]:
num_duplicates = df.duplicated().sum()
print("Number of duplicate rows found:", num_duplicates)


Number of duplicate rows found: 0


**How `df.duplicated()` works internally:**
- It compares **every row** against **all previous rows**.
- For each row, it returns `True` if an identical row (same value in *every*
  single column) has already appeared earlier in the DataFrame, and `False`
  otherwise (including the very first occurrence, which is never marked as
  a duplicate).
- `.sum()` then just adds up how many `True` values there are, i.e. how many
  duplicate rows exist.

**Result on this dataset:** `0` — BankChurners.csv has no duplicate
customer records. This is good news, but I still need to *show* the
mechanism actually works, not just report a zero. So I'll simulate a
"dirty" version of the data below.

## Step 3: Demonstrate the Technique with Injected Duplicates
(This is only for demonstration — it does **not** touch the real
preprocessing pipeline. I copy the first 3 rows and append them again to a
**separate copy** of the DataFrame, `df_demo`, to simulate what a messy,
real-world dataset with duplicate entries would look like.)

In [3]:
df_demo = pd.concat([df, df.iloc[0:3]], ignore_index=True)
print("Demo shape after injecting 3 duplicate rows:", df_demo.shape)
print("(Original was", df.shape, "— so 3 extra rows were added)")


Demo shape after injecting 3 duplicate rows: (10130, 23)
(Original was (10127, 23) — so 3 extra rows were added)


## Step 4: Detect the Injected Duplicates

In [4]:
duplicate_flags = df_demo.duplicated()
print("Duplicate flags (True = duplicate row):")
print(duplicate_flags.value_counts())

print("\nNumber of duplicate rows detected:", duplicate_flags.sum())


Duplicate flags (True = duplicate row):
False    10127
True         3
Name: count, dtype: int64

Number of duplicate rows detected: 3


**Explanation for the examiner:**
- `duplicate_flags` is a Boolean Series the same length as `df_demo`.
- The **last 3 rows** (the ones I copied and appended) are marked `True`,
  because pandas found an identical row earlier in the table.
- The original first 3 rows stay `False`, because they were the *first*
  occurrence — pandas never flags the original, only the repeats.

## Step 5: View the Actual Duplicate Rows

In [5]:
duplicate_rows = df_demo[df_demo.duplicated()]
print("These are the rows pandas identified as duplicates:")
duplicate_rows


These are the rows pandas identified as duplicates:


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
10127,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
10128,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
10129,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998


**What this does:** `df_demo[df_demo.duplicated()]` filters the
DataFrame down to *only* the rows flagged `True` above, so we can visually
confirm they are indeed exact copies of earlier rows (same `CLIENTNUM`,
same `Customer_Age`, same everything).

## Step 6: Remove the Duplicate Rows

In [6]:
print("Shape before removing duplicates:", df_demo.shape)

df_demo_cleaned = df_demo.drop_duplicates()

print("Shape after removing duplicates:", df_demo_cleaned.shape)
print("Duplicate rows remaining:", df_demo_cleaned.duplicated().sum())


Shape before removing duplicates: (10130, 23)
Shape after removing duplicates: (10127, 23)
Duplicate rows remaining: 0


**How `drop_duplicates()` works:** It keeps the **first occurrence** of
every row and deletes every later row that is an exact match — which is
exactly the 3 injected rows we added. The shape drops back down to the
original `df.shape`, confirming the cleanup worked correctly.

## Step 7: Apply to the Real Pipeline
Now, back on the **actual** dataset (not the demo copy) — since we already
confirmed 0 duplicates exist, this line is a safety net: if the raw CSV
ever changes and duplicate rows appear, this line will still catch and
remove them automatically.

In [7]:
print("Duplicate rows in real dataset:", df.duplicated().sum())
df = df.drop_duplicates()
print("Final shape after duplicate check:", df.shape)


Duplicate rows in real dataset: 0
Final shape after duplicate check: (10127, 23)


## Summary — Talking Points for Presentation

| Question | Answer |
|---|---|
| **What is a duplicate row?** | A row where *every single column value* exactly matches an earlier row — e.g. the same customer record appearing twice. |
| **Why check for it?** | If left in, duplicate rows quietly give one customer extra influence during model training, biasing results toward them. |
| **How does `df.duplicated()` work?** | Scans row by row; marks a row `True` only if an identical row already appeared earlier — the original copy is never flagged. |
| **How does `df.drop_duplicates()` work?** | Keeps the first occurrence of each row, deletes any exact repeats that come after it. |
| **What did we find in BankChurners.csv?** | Zero duplicate rows — the dataset was already clean on this front. |
| **Why demonstrate it anyway?** | To prove the detection/removal logic genuinely works, not just to report an untested "0" — this was shown by temporarily injecting 3 copied rows, detecting them, and removing them back down to the original shape. |
